# Supply Chain Data Analyst — тестовое задание (Генерация)

Расчёт рекомендуемого количества к заказу (`RecommendedOrder`) по 200 SKU на дату 19.09.2025.

## Настройка

Импорты и настраиваемые параметры — общие для всего ноутбука. Все новые импорты добавляются в ячейку ниже, все новые параметры — в ячейку после неё, а не в код по ходу работы.

In [1]:
# Path — для удобной и переносимой работы с файловыми путями.
from pathlib import Path

# pandas — основной инструмент для чтения Excel и работы с таблицами.
import pandas as pd

In [2]:
# Имена файлов с исходными данными — если файлы переименуют, меняем только здесь.
CANDIDATE_FILE_NAME = "candidate_data.xlsx"
DICTIONARY_FILE_NAME = "data_dictionary.xlsx"

# Раздел 1. Baseline

## Шаг 1. Пути к данным

Определяем расположение исходных файлов (`candidate_data.xlsx`, `data_dictionary.xlsx`) относительно ноутбука и проверяем, что они на месте.

In [3]:
# Ноутбук лежит в notebooks/, поэтому поднимаемся на один уровень вверх, чтобы попасть в корень проекта.
project_dir = Path.cwd().parent

# Папка с исходными данными — на уровне корня проекта, не внутри notebooks/.
data_dir = project_dir / "data"

# Полные пути к обоим Excel-файлам, собранные из констант в ячейке параметров.
candidate_path = data_dir / CANDIDATE_FILE_NAME
dictionary_path = data_dir / DICTIONARY_FILE_NAME

# Проверяем, что оба файла реально существуют по этим путям, прежде чем пытаться их читать.
# Если assert сработает — сразу понятно, что не так, вместо непонятной ошибки на этапе чтения Excel.
assert candidate_path.exists(), f"Не найден файл: {candidate_path}"
assert dictionary_path.exists(), f"Не найден файл: {dictionary_path}"

print("candidate_data.xlsx найден:", candidate_path)
print("data_dictionary.xlsx найден:", dictionary_path)

candidate_data.xlsx найден: C:\Users\User\Desktop\clode folder\projects_code\тестовое от Генерации\reorder-model\data\candidate_data.xlsx
data_dictionary.xlsx найден: C:\Users\User\Desktop\clode folder\projects_code\тестовое от Генерации\reorder-model\data\data_dictionary.xlsx


## Шаг 2. Структура Excel-файлов

Смотрим список листов в обоих файлах, не загружая данные целиком.

In [4]:
# pd.ExcelFile открывает файл и читает его оглавление (список листов),
# но не загружает содержимое листов в память — это быстрее, чем сразу читать все данные.
candidate_excel = pd.ExcelFile(candidate_path)
dictionary_excel = pd.ExcelFile(dictionary_path)

print("Листы в candidate_data.xlsx:", candidate_excel.sheet_names)
print("Листы в data_dictionary.xlsx:", dictionary_excel.sheet_names)

Листы в candidate_data.xlsx: ['SKU', 'DailyHistory', 'OpenSupply', 'OrderCalendar']
Листы в data_dictionary.xlsx: ['Поля', 'Условия']


## Шаг 3. Словарь полей

Читаем лист "Поля" — описание всех колонок в данных, чтобы дальше корректно их интерпретировать.

In [5]:
# Читаем лист "Поля" целиком — он небольшой, это справочная таблица, а не данные для расчёта.
fields_df = pd.read_excel(dictionary_path, sheet_name="Поля")

# pd.set_option, чтобы длинные текстовые описания не обрезались "...".
pd.set_option("display.max_colwidth", None)

fields_df

,Лист,Поле,Единицы,Определение
0,SKU,SKU,Текст,Идентификатор позиции активного ассортимента. Один SKU — одна строка.
1,SKU,CurrentStock,Шт.,"Текущий остаток на дату расчёта 19.09.2025, до размещения заказа."
2,SKU,InTransit,Шт.,"Отгруженное, ещё не полученное количество. Сумма строк OpenSupply со Status = InTransit."
3,SKU,OpenPO,Шт.,"Размещённое, ещё не отгруженное количество. Сумма строк OpenSupply со Status = OpenPO."
4,SKU,LeadTimeDays,Календарные дни,Учебный срок нового заказа — 10 дней. Для уже размещённых поставок используйте ExpectedReceiptDate.
5,DailyHistory,Date,Дата,День наблюдения. Одна строка на SKU и календарный день.
6,DailyHistory,SKU,Текст,"Идентификатор, связанный с листом SKU."
7,DailyHistory,SalesQty,Шт.,Фактически выполненные продажи за указанный день. Возвраты не вычитаются.
8,DailyHistory,Остаток на конец дня,Шт.,Остаток на конец указанного дня. Пустая ячейка означает отсутствие наблюдения.
9,OpenSupply,PO_ID,Текст,Обезличенный идентификатор отдельной учебной поставки.


## Шаг 4. Условия задачи

Читаем лист "Условия" — явные допущения и параметры, заданные заказчиком (срок поставки, календарь и т.п.).

In [6]:
conditions_df = pd.read_excel(dictionary_path, sheet_name="Условия")
conditions_df

,Параметр,Значение
0,Дата расчёта,"19.09.2025, до размещения заказа"
1,История,19.06.2024–18.09.2025; 457 дней; 200 SKU
2,Гранулярность,SKU × календарный день
3,Конец дня,Столбец «Остаток на конец дня» относится к дню Date.
4,Единицы,Все количества — штуки; внутренние перемещения возможны поштучно.
5,Отсутствующие значения,Пустой остаток — наблюдения нет. Все SKU активного ассортимента остаются в итоговом расчёте.
6,История продаж,Фактически выполненные продажи. Исторического журнала неудовлетворённого спроса нет.
7,Учебные условия,"LeadTimeDays, календарь заказов и строки OpenSupply заданы для задания; это не реальные параметры компании."
8,Поставки,InTransit и OpenPO — разные состояния. Сводные количества SKU и строки OpenSupply описывают один набор поставок.
9,Исторические поставки,OpenSupply — снимок только на дату расчёта. Для backtesting требуется явно задать начальное состояние и правила моделирования.


## Шаг 5. Лист SKU

Загружаем таблицу активного ассортимента: текущие остатки, открытые поставки, срок поставки.

In [7]:
# Лист SKU — одна строка на позицию активного ассортимента (ожидаем 200 строк).
sku_df = pd.read_excel(candidate_path, sheet_name="SKU")

print("Форма таблицы (строки, колонки):", sku_df.shape)
print("Колонки:", list(sku_df.columns))
sku_df.head()

Форма таблицы (строки, колонки): (200, 5)
Колонки: ['SKU', 'CurrentStock', 'InTransit', 'OpenPO', 'LeadTimeDays']


,SKU,CurrentStock,InTransit,OpenPO,LeadTimeDays
0,TEST-0001,0,0,0,10
1,TEST-0002,3,0,1,10
2,TEST-0003,19,4,7,10
3,TEST-0004,0,0,0,10
4,TEST-0005,18,2,0,10


## Шаг 6. Лист DailyHistory

Загружаем дневную историю продаж и остатков — самая большая таблица (200 SKU × 457 дней).

In [8]:
history_df = pd.read_excel(candidate_path, sheet_name="DailyHistory")

print("Форма таблицы (строки, колонки):", history_df.shape)
print("Колонки:", list(history_df.columns))
print("Диапазон дат:", history_df["Date"].min(), "—", history_df["Date"].max())
history_df.head()

Форма таблицы (строки, колонки): (91400, 4)
Колонки: ['Date', 'SKU', 'SalesQty', 'Остаток на конец дня']
Диапазон дат: 2024-06-19 00:00:00 — 2025-09-18 00:00:00


,Date,SKU,SalesQty,Остаток на конец дня
0,2024-06-19,TEST-0001,0,2.0
1,2024-06-20,TEST-0001,0,2.0
2,2024-06-21,TEST-0001,0,2.0
3,2024-06-22,TEST-0001,0,2.0
4,2024-06-23,TEST-0001,0,2.0


## Шаг 7. Лист OpenSupply

Загружаем список уже размещённых поставок (InTransit/OpenPO) — снимок на дату расчёта.

In [9]:
open_supply_df = pd.read_excel(candidate_path, sheet_name="OpenSupply")

print("Форма таблицы (строки, колонки):", open_supply_df.shape)
print("Колонки:", list(open_supply_df.columns))
open_supply_df.head()

Форма таблицы (строки, колонки): (199, 6)
Колонки: ['PO_ID', 'SKU', 'Qty', 'Status', 'PlacedDate', 'ExpectedReceiptDate']


,PO_ID,SKU,Qty,Status,PlacedDate,ExpectedReceiptDate
0,PO-0002-1,TEST-0002,1,OpenPO,2025-09-15,2025-09-24
1,PO-0003-1,TEST-0003,4,InTransit,2025-09-14,2025-09-27
2,PO-0003-2,TEST-0003,7,OpenPO,2025-09-13,2025-10-04
3,PO-0005-1,TEST-0005,2,InTransit,2025-09-12,2025-10-03
4,PO-0006-1,TEST-0006,1,OpenPO,2025-09-11,2025-10-06


## Шаг 8. Лист OrderCalendar

Загружаем календарь размещения заказов — понадобится для расчёта protection period.

In [10]:
order_calendar_df = pd.read_excel(candidate_path, sheet_name="OrderCalendar")

print("Форма таблицы (строки, колонки):", order_calendar_df.shape)
print("Колонки:", list(order_calendar_df.columns))
order_calendar_df.head()

Форма таблицы (строки, колонки): (157, 3)
Колонки: ['OrderDate', 'NextOrderDate', 'OrderCycleDays']


,OrderDate,NextOrderDate,OrderCycleDays
0,2024-06-19,2024-06-21,2
1,2024-06-21,2024-06-26,5
2,2024-06-26,2024-06-28,2
3,2024-06-28,2024-07-03,5
4,2024-07-03,2024-07-05,2


## Шаг 9. Полнота календарной истории

Проверяем, что в `DailyHistory` ровно одна строка на каждую пару SKU-день, без пропущенных дней и без дублей.

In [11]:
# Считаем число строк, число уникальных пар SKU+Date и ожидаемое число (SKU x дни).
n_rows = len(history_df)
n_unique_pairs = history_df[["SKU", "Date"]].drop_duplicates().shape[0]
n_expected = sku_df["SKU"].nunique() * history_df["Date"].nunique()

print("Строк в DailyHistory:", n_rows)
print("Уникальных пар SKU+Date:", n_unique_pairs)
print("Ожидается (SKU x дни):", n_expected)
print("Дубликатов пар SKU+Date:", n_rows - n_unique_pairs)

Строк в DailyHistory: 91400
Уникальных пар SKU+Date: 91400
Ожидается (SKU x дни): 91400
Дубликатов пар SKU+Date: 0


## Шаг 10. Пропуски в данных

Считаем долю пропусков в `SalesQty` и в «Остаток на конец дня» (по словарю полей — пустой остаток значит «нет наблюдения», а не ноль).

In [12]:
missing_sales = history_df["SalesQty"].isna().sum()
missing_stock = history_df["Остаток на конец дня"].isna().sum()

print(f"Пропуски в SalesQty: {missing_sales} ({missing_sales / n_rows:.2%})")
print(f"Пропуски в 'Остаток на конец дня': {missing_stock} ({missing_stock / n_rows:.2%})")

Пропуски в SalesQty: 0 (0.00%)
Пропуски в 'Остаток на конец дня': 1489 (1.63%)


## Шаг 11. Активность продаж по SKU

Для каждого SKU считаем долю дней с продажами и суммарный спрос — чтобы понять, насколько распространён прерывистый спрос и есть ли SKU совсем без истории продаж.

In [13]:
# Группируем по SKU и считаем базовые показатели активности спроса.
sku_activity = history_df.groupby("SKU").agg(
    days_total=("SalesQty", "size"),
    days_with_sales=("SalesQty", lambda s: (s > 0).sum()),
    total_sales=("SalesQty", "sum"),
    mean_daily_sales=("SalesQty", "mean"),
).reset_index()

# Доля дней с продажами — ключевой индикатор прерывистости спроса.
sku_activity["share_days_with_sales"] = sku_activity["days_with_sales"] / sku_activity["days_total"]

print("SKU без единой продажи за всю историю:", (sku_activity["total_sales"] == 0).sum())
print("Медианная доля дней с продажами по SKU:", round(sku_activity["share_days_with_sales"].median(), 4))
print("SKU с долей дней с продажами < 10%:", (sku_activity["share_days_with_sales"] < 0.10).sum())

sku_activity.sort_values("total_sales").head(10)

SKU без единой продажи за всю историю: 18
Медианная доля дней с продажами по SKU: 0.0098
SKU с долей дней с продажами < 10%: 170


,SKU,days_total,days_with_sales,total_sales,mean_daily_sales,share_days_with_sales
7,TEST-0008,457,0,0,0.0,0.0
11,TEST-0012,457,0,0,0.0,0.0
29,TEST-0030,457,0,0,0.0,0.0
18,TEST-0019,457,0,0,0.0,0.0
49,TEST-0050,457,0,0,0.0,0.0
59,TEST-0060,457,0,0,0.0,0.0
43,TEST-0044,457,0,0,0.0,0.0
34,TEST-0035,457,0,0,0.0,0.0
60,TEST-0061,457,0,0,0.0,0.0
77,TEST-0078,457,0,0,0.0,0.0


## Шаг 12. Проверка на очевидные аномалии

Ищем отрицательные значения продаж/остатков и другие явно некорректные данные.

In [14]:
print("Отрицательные значения SalesQty:", (history_df["SalesQty"] < 0).sum())
print("Отрицательные значения остатка:", (history_df["Остаток на конец дня"] < 0).sum())
print("Максимальная продажа за день:", history_df["SalesQty"].max())
print("Максимальный остаток на конец дня:", history_df["Остаток на конец дня"].max())

Отрицательные значения SalesQty: 0
Отрицательные значения остатка: 0
Максимальная продажа за день: 150
Максимальный остаток на конец дня: 559.0


### Выводы и наблюдения

- Календарь наблюдений полный: 91 400 строк, дублей нет, пропущенных дней нет.
- `SalesQty` заполнен без единого пропуска — для наивной оценки спроса дополнительная очистка не требуется.
- В «Остаток на конец дня» — 1,63% пропусков (нет наблюдения, не ноль). Для baseline не критично (спрос оцениваем по `SalesQty`), но важно для бэктеста в разделе 2, где понадобится стартовое состояние запасов.
- Спрос по большинству SKU сильно прерывистый: медианная доля дней с продажами — **0,98%**, у 170 из 200 SKU (85%) доля дней с продажами меньше 10%. Наивное среднее по всей истории это переживёт, но будет давать грубую оценку — это ожидаемая точка для доработки в разделе 2, а не проблема, которую нужно решать прямо сейчас.
- **18 SKU (9%) не имеют вообще ни одной продажи за все 457 дней.** Для них наивная оценка спроса даст 0 — понадобится явный fallback и пометка в итоговой таблице (по условию задания такие позиции нельзя просто пропустить).
- Отрицательных значений и явных выбросов не найдено (максимум продаж за день — 150 шт., максимум остатка — 559 шт.) — данные в этой части чистые.

**Отдельный шаг «приведение данных в порядок» пропускаем** — для baseline он оказался бы пустым: `SalesQty` без пропусков/дублей/отрицательных значений, а два найденных момента (пропуски остатка, SKU без истории) — это не грязные данные, а два случая, которые нужно явно обработать логикой модели, а не почистить заранее.

# Раздел 2. Бэктесты и отладка

# Раздел 3. Заказ